# Scenario Dreamer Diversity Audit (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/scenario-dreamer-decision-layer/blob/main/notebooks/scenario_dreamer_diversity_audit_colab.ipynb)

This notebook runs **Experiment 0**: hold one fixed scenario constant and rerun the same stock frozen baseline under multiple seeds.

Goal:
- decide whether the current frozen stack exhibits enough branching to justify a distribution-aware selector.


In [ ]:
# Step 1: Sync this repo into the Colab runtime
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/scenario-dreamer-decision-layer.git'
REPO_DIR = '/content/scenario-dreamer-decision-layer'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'src')):
    if p not in sys.path:
        sys.path.insert(0, p)

print('repo_rev:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Suggested defaults for Experiment 0
%env SCENARIO_DREAMER_DRIVE_ROOT=/content/drive/MyDrive/scenario_dreamer_research
%env SCENARIO_DREAMER_RUN_SETUP=1
%env SCENARIO_DREAMER_AUDIT_SCENARIO_INDEX=0
%env SCENARIO_DREAMER_AUDIT_SEEDS=0,1,2,3
%env SCENARIO_DREAMER_AUDIT_VISUALIZE=0
%env SCENARIO_DREAMER_AUDIT_NO_LIGHTWEIGHT=0


In [ ]:
# Step 2: Mount Drive, clone/pin upstream Scenario Dreamer, and bind Drive-backed assets/results
import json
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR], check=True)

from scenario_dreamer_decision_layer.colab import bind_drive_layout, inspect_bound_layout

subprocess.run([sys.executable, 'scripts/bootstrap_baseline.py', '--clone-upstream', '--write-lock'], check=True)
binding = bind_drive_layout(os.environ['SCENARIO_DREAMER_DRIVE_ROOT'])
print('binding:')
print(json.dumps(binding, indent=2, sort_keys=True))
print('inspection:')
print(json.dumps(inspect_bound_layout(), indent=2, sort_keys=True))


In [ ]:
# Step 3: Install a lean runtime for Scenario Dreamer simulation on Colab
import os
import subprocess
import sys

RUN_SETUP = os.environ.get('SCENARIO_DREAMER_RUN_SETUP', '1').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
print('RUN_SETUP:', RUN_SETUP)
if RUN_SETUP:
    subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py', '--editable-project'], check=True)
else:
    print('Skipping runtime setup.')


In [ ]:
# Step 4: Confirm assets and scenario availability before the audit
import json
import subprocess
import sys

raw = subprocess.check_output([sys.executable, 'scripts/run_smoke_eval.py', '--dry-run'], text=True)
print(raw)
smoke_dry_run = json.loads(raw)
missing_assets = smoke_dry_run.get('missing_assets', {})
scenario_count = int(smoke_dry_run.get('scenario_count', 0))
print('preflight_summary:')
print(json.dumps({
    'scenario_count': scenario_count,
    'missing_assets': missing_assets,
    'config_snapshot': smoke_dry_run.get('config_snapshot', ''),
}, indent=2, sort_keys=True))
if missing_assets:
    raise RuntimeError(f'Resolve missing assets before Step 5: {missing_assets}')
if scenario_count <= 0:
    raise RuntimeError('Preflight found zero scenarios. Verify the Scenario Dreamer environment pack binding before Step 5.')


In [ ]:
# Step 5: Run Experiment 0 on one fixed scenario across multiple seeds
import json
import os
import subprocess
import sys


def _run_json_cmd(cmd, label):
    proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
    stdout = proc.stdout.strip()
    stderr = proc.stderr.strip()
    if stdout:
        print(stdout)
    if stderr:
        print(f'[{label}_stderr]')
        print(stderr)
    if proc.returncode != 0:
        diag = {
            'exit_code': proc.returncode,
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    try:
        return json.loads(stdout)
    except json.JSONDecodeError as exc:
        diag = {
            'failed_command': ' '.join(cmd),
            'results_root': os.environ.get('SCENARIO_DREAMER_RESULTS_ROOT', ''),
            'recent_stdout': stdout[-4000:],
            'recent_stderr': stderr[-4000:],
            'parse_error': str(exc),
        }
        print(f'{label}_diagnostics:')
        print(json.dumps(diag, indent=2, sort_keys=True))
        raise

scenario_index = int(os.environ.get('SCENARIO_DREAMER_AUDIT_SCENARIO_INDEX', '0'))
seeds = os.environ.get('SCENARIO_DREAMER_AUDIT_SEEDS', '0,1,2,3').strip()
visualize = os.environ.get('SCENARIO_DREAMER_AUDIT_VISUALIZE', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}
no_lightweight = os.environ.get('SCENARIO_DREAMER_AUDIT_NO_LIGHTWEIGHT', '0').strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

cmd = [
    sys.executable,
    'scripts/run_diversity_audit.py',
    '--scenario-index',
    str(scenario_index),
    '--seeds',
    seeds,
]
if visualize:
    cmd.append('--visualize')
if no_lightweight:
    cmd.append('--no-lightweight')

audit_payload = _run_json_cmd(cmd, 'step5_diversity_audit')
print('diversity_audit_summary:')
print(json.dumps({
    'run_id': audit_payload.get('run_id', ''),
    'run_dir': audit_payload.get('run_dir', ''),
    'scenario_name': audit_payload.get('scenario_name', ''),
    'seeds': audit_payload.get('seeds', []),
    'diversity_detected': audit_payload.get('diversity_detected', False),
    'decision': audit_payload.get('decision', ''),
    'metric_spread': audit_payload.get('metric_spread', {}),
}, indent=2, sort_keys=True))


In [ ]:
# Step 6: Inspect the saved audit summary and per-seed metrics
import json
import os
from pathlib import Path

results_root = Path(os.environ['SCENARIO_DREAMER_RESULTS_ROOT'])
run_dirs = sorted([p for p in results_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
print('results_root:', results_root)
print('num_run_dirs:', len(run_dirs))
if not run_dirs:
    raise RuntimeError('No run directories found. Execute Step 5 first.')

latest = run_dirs[0]
print('latest_run_dir:', latest)
summary_path = latest / 'diversity_audit_summary.json'
if not summary_path.exists():
    raise RuntimeError(f'Latest run does not contain diversity_audit_summary.json: {latest}')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2, sort_keys=True)[:12000])

for seed_dir in sorted([p for p in latest.iterdir() if p.is_dir() and p.name.startswith('seed_')]):
    metrics_path = seed_dir / 'metrics.json'
    if metrics_path.exists():
        print(seed_dir.name, metrics_path.read_text())
